In [1]:
# Imports

import os
import glob
import pandas as pd 
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from scipy.signal import find_peaks

In [20]:
# Configuration

USE_DUMMY_DATA = False

EXPORT_TO_CSV = True
overwrite = None    # None = no decision made, True = overwrite all, False = skip all

LEAD_IN_METERS = 20.0
FOLLOW_THROUGH_METERS = 2.0
MIN_EVENT_DISTANCE_M = 50.0
PROBABILITY_THRESHOLD = 0.65

In [21]:
# Helper functions

# calculates geometric features from coordinates
def calculate_geometric_features(df_coords):
    points = df_coords[['x', 'y', 'z']].to_numpy()
    distances = np.linalg.norm(np.diff(points, axis=0), axis=1)
    cumulative_distance = np.insert(np.cumsum(distances), 0, 0)
    df_coords['Distance'] = cumulative_distance

    dx = np.gradient(df_coords['x'])
    ddx = np.gradient(dx)
    dy = np.gradient(df_coords['y'])
    ddy = np.gradient(dy)
    curvature = np.abs(dx * ddy - dy * ddx) / np.power(dx**2 + dy**2, 1.5)
    df_coords['curvature'] = curvature

    df_coords['elevation_change'] = np.gradient(df_coords['z'])

    window_size = 15
    df_coords['curvature_smooth'] = df_coords['curvature'].rolling(window=window_size, center=True, min_periods=1).mean()
    df_coords['elevation_change_smooth'] = df_coords['elevation_change'].rolling(window=window_size, center=True, min_periods=1).mean()

    df_coords.fillna(0, inplace=True)
    return df_coords

# Loads data for known track and creates a target variable
def prepare_training_data_events(coord_path, sections_path, track_name):
    try:
        df_coords = pd.read_csv(coord_path)
        df_sections = pd.read_csv(sections_path)
        df_sections['start_m']   = pd.to_numeric(df_sections['start_m'], errors='coerce')
        df_sections['ed_m']   = pd.to_numeric(df_sections['end_m'], errors='coerce')
        df_sections.dropna(subset=['start_m', 'end_m'], inplace=True)

    except FileNotFoundError as e:
        print(f"   [Error] Could not find file for track '{track_name}': {e}. Skipping this track.")
        return None
    
    df_train = calculate_geometric_features(df_coords)
    df_train['track_name'] = track_name
    df_train['is_section_start'] = 0
    for _, row in df_sections.iterrows():
        start_m = row['start_m']
        start_of_zone = max(0, start_m - LEAD_IN_METERS)
        end_of_zone = start_m + FOLLOW_THROUGH_METERS
        zone_indices = (df_train['Distance'] >= start_of_zone) & (df_train['Distance'] < end_of_zone)   # data point indices in region
        df_train.loc[zone_indices, 'is_section_start'] = 1 # label these points as 1
    
    return df_train

# finds peaks in the probability signal
def probabilities_to_sections(df_with_probs, min_event_distance_m, probability_threshold):
    probs = df_with_probs['start_probability'].to_numpy()
    distances = df_with_probs['Distance'].to_numpy()

    # find average distance in meters to convert to indices
    avg_dist_per_point = np.mean(np.diff(distances))
    min_dist_indices = int(min_event_distance_m / avg_dist_per_point)

    peak_indices, _ = find_peaks(probs, height=probability_threshold, distance=min_dist_indices)  # height and distance are minimum probability and distance between peaks
    if len(peak_indices) == 0:
        print('Warning: No section starts detected. Try lowering the peak height threshold.')
        return pd.DataFrame(columns=['segment_id', 'start_m', 'end_m'])
    section_starts = distances[peak_indices]    # the start of each section is the distance at each peak
    final_starts = np.insert(section_starts, 0, 0.0)    # first section starts at 0.0
    final_starts = np.unique(final_starts)  # removes duplicates
    sections = []
    for i in range(len(final_starts)):
        start_m = final_starts[i]
        end_m = final_starts[i+1] if i+1 < len(final_starts) else distances[-1]
        sections.append({
            'segment_id': i,
            'start_m': start_m,
            'end_m': end_m
        })
    return pd.DataFrame(sections)


In [23]:
# Execution

def main():

    # set up paths
    data_type = "Dummy" if USE_DUMMY_DATA else "Real"
    base_path = os.getcwd()
    coord_base_path = os.path.join(base_path, "Coordinates")
    sections_base_path = os.path.join(base_path, "Sections", data_type)
    print(f"\n--- Using {data_type.upper()} data, read from subfolders in: {base_path}. ---")

    # discover training and target tracks
    print("\n--- Discovering Tracks ---")
    all_coord_files = glob.glob(os.path.join(coord_base_path, "*.csv"))
    all_track_names = {os.path.basename(f).replace('.csv','')  for f in all_coord_files}
    section_files = glob.glob(os.path.join(sections_base_path, "sections_*.csv"))
    training_track_names = {os.path.basename(f).replace('sections_', '').replace('.csv', '') for f in section_files}
    target_track_names = list(all_track_names - training_track_names)

    if not training_track_names:
        print(f"\n[FATAL ERROR] No training tracks found in '{sections_base_path}'.")
        print(" A 'sections_{track_name}.csv' file is required for at least one track to train the model.")
        return
    
    if not target_track_names:
        print(f"\n[FATAL ERROR] No target tracks found.")
        print(" A target tarck is a .csv file in the 'Coordinates' folder that does not have a corresponding 'sections_*.csv' file.")
        return
    
    print(f"Found {len(training_track_names)} tracks for training: {sorted(list(training_track_names))}")
    print(f"Found {len(target_track_names)} tracks for prediction: {sorted(list(training_track_names))}")

    # load all data
    print("\n--- Loading and Preparing ALL Training Data ---")
    all_training_data = []
    for track_name in sorted(list(training_track_names)):
        print(f"Processing training track: '{track_name}'...")
        coord_path = os.path.join(coord_base_path, f"{track_name}.csv")
        sections_path = os.path.join(sections_base_path, f"sections_{track_name}.csv")
        track_df = prepare_training_data_events(coord_path, sections_path, track_name)
        if track_df is not None:
            all_training_data.append(track_df)
        
    if not all_training_data:
        print("\n[FATAL ERROR] No valid training data could be loaded. Exiting.")
        return
    
    master_training_df = pd.concat(all_training_data, ignore_index=True)
    print("All track data has been loaded and concatenated.")

    # training master model
    print("\n--- Training Master Model ---")
    features = ['curvature', 'elevation_change', 'curvature_smooth', 'elevation_change_smooth']
    X_train = master_training_df[features]
    y_train = master_training_df['is_section_start']

    model = RandomForestClassifier(n_estimators=100, random_state=43, class_weight='balanced', n_jobs=-1)
    model.fit(X_train, y_train)
    print("Master model trained successfully on all available tracks.")

    overwrite_all = overwrite

    # predict on target tracks
    print("\n--- Starting Prediction on Target Tracks ---")
    for target_track in sorted(target_track_names):
        print(f"\n>> Predicting on Target Track: '{target_track.upper()}' <<")
        target_coord_path = os.path.join(coord_base_path, f"{target_track}.csv")
        try:
            predict_df = pd.read_csv(target_coord_path)
        except FileNotFoundError:
            print(f"   [Error] Could not find coordinate file: {target_coord_path}. Skipping.")
            continue

        predict_df = calculate_geometric_features(predict_df)
        X_predict = predict_df[features]
        print("   Geometric features calculated.")

        probabilities = model.predict_proba(X_predict)[:, 1]
        predict_df['start_probability'] = probabilities
        print("   Prediction complete.")

        final_sections = probabilities_to_sections(predict_df, MIN_EVENT_DISTANCE_M, PROBABILITY_THRESHOLD)
        print(f"   Found {len(final_sections)} sections for '{target_track}'.")

        predict_df.to_csv(os.path.join(sections_base_path, f"sections_debug_{target_track}.csv"), index=False)

        # export
        if EXPORT_TO_CSV:
            output_filename = f"sections_target_{target_track}.csv"
            output_path = os.path.join(sections_base_path, output_filename)
            if os.path.exists(output_path):
                if overwrite_all is None:
                    choice = input(f"Output file '{output_filename}' exists. Overwrite this and ALL subsequent existing file? (y/n): ").lower()
                    overwrite_all = (choice in ['y', 'yes'])
                if overwrite_all:
                    final_sections.to_csv(output_path, index=False)
                    print(f"   File Overwritten: {output_path}")
                else:
                    print(f"   Skipped existing file: {output_path}")
            else:
                final_sections.to_csv(output_path, index=False)
                print(f"   File exported: {output_path}")
    
    
    print("\n--- All processing complete. ---")


if __name__ == "__main__":
    main()


--- Using REAL data, read from subfolders in: h:\My Drive\Cranfield\11. Individual Thesis\Code. ---

--- Discovering Tracks ---
Found 7 tracks for training: ['brands_gp', 'donington_gp', 'hungaroring', 'silverstone_gp', 'spa', 'target_monza', 'zandvoort']
Found 1 tracks for prediction: ['brands_gp', 'donington_gp', 'hungaroring', 'silverstone_gp', 'spa', 'target_monza', 'zandvoort']

--- Loading and Preparing ALL Training Data ---
Processing training track: 'brands_gp'...
Processing training track: 'donington_gp'...
Processing training track: 'hungaroring'...
Processing training track: 'silverstone_gp'...
Processing training track: 'spa'...
Processing training track: 'target_monza'...
   [Error] Could not find file for track 'target_monza': [Errno 2] No such file or directory: 'h:\\My Drive\\Cranfield\\11. Individual Thesis\\Code\\Coordinates\\target_monza.csv'. Skipping this track.
Processing training track: 'zandvoort'...
All track data has been loaded and concatenated.

--- Trainin